# Data Migration from CSV files to PostgreSQL database

The following code is used to load bootcamp and cost of living data from local `.csv` files directly into the database defined in the application.

In [33]:
import pandas as pd
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path (to enable imports from utils)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.db_handler import DatabaseHandler

In [34]:
# Load environment variables
env_path = os.path.join('..', '.env')
if not os.path.exists(env_path):
    env_path = os.path.join('..', '.env.example')
load_dotenv(env_path)

# Build database URL for local environment (using localhost instead of docker service name)
db_user = os.getenv('POSTGRES_USER')
db_password = os.getenv('POSTGRES_PASSWORD')
db_name = os.getenv('POSTGRES_DB')
db_port = os.getenv('POSTGRES_PORT')
db_host = 'localhost'

database_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# Initialize database connection
db = DatabaseHandler(database_url=database_url)

In [35]:
# Define paths to CSV files - assuming files are located in the project root folder
bootcamps_csv = '../examples/Bootcamps.xlsx'
living_costs_csv = '../examples/Living_costs.csv'

In [36]:
def load_csv_data_to_postgres():
    # 1. Bootcamp data
    try:
        df_bootcamp = pd.read_excel(bootcamps_csv)
        # Save to database in 'bootcamp_data' table
        df_bootcamp.to_sql('bootcamp_data', con=db.engine, if_exists='replace', index=False)
        print(f"Saved bootcamp data ({len(df_bootcamp)} rows) to table 'bootcamp_data'.")
    except FileNotFoundError:
        print(f"File {bootcamps_csv} not found. Please ensure it exists in the specified location.")
    except Exception as e:
        print(f"An error occurred while loading bootcamp data: {e}")
        
    # 2. Living costs data
    try:
        df_koszty = pd.read_csv(living_costs_csv, sep=',')
        # Save to database in 'living_costs_data' table
        df_koszty.to_sql('living_costs_data', con=db.engine, if_exists='replace', index=False)
        print(f"Saved living costs data ({len(df_koszty)} rows) to table 'living_costs_data'.")
    except FileNotFoundError:
        print(f"File {living_costs_csv} not found. Please ensure it exists in the specified location.")
    except Exception as e:
        print(f"An error occurred while loading living costs data: {e}")

load_csv_data_to_postgres()

Saved bootcamp data (27 rows) to table 'bootcamp_data'.
Saved living costs data (16 rows) to table 'living_costs_data'.
